<a href="https://colab.research.google.com/github/postnicov/ResazurinResorufin/blob/main/Munsell_C_to_Lab_D65_to_Munsell_C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Munsell C to CIE Lab D65 to Munsell C

**Version 1.1.0**

This Colab-compatible notebook converts a Munsell notation represented in the standard Illuminant C renotation system into CIE Lab relative to D65. It then treats the Lab triplet as a new input and restores the Munsell notation under Illuminant C.

The route is: Munsell(C) -> xyY(C) -> XYZ(C) -> adapted XYZ(D65) -> Lab(D65) -> XYZ(D65) -> adapted XYZ(C) -> xyY(C) -> Munsell(C).

Version 1.1.0 changes:

- The first executable cell checks for required packages, installs missing packages only, and imports all libraries used by the notebook.
- The restored-hue rule converts 0.0 of a sector to 10 of the preceding sector while preserving value and chroma.
- All executable cells have Google Colab titles using #@title.
- The notebook uses compatibility-safe colour-science adaptation and Delta E APIs.

In [1]:
#@title 1. Check dependencies, install if needed, and import libraries
# This cell checks the Python environment before installing anything.
# NumPy is normally preinstalled in Colab; colour-science may not be.
import sys
import subprocess
import importlib.util

required_packages = {
    'numpy': 'numpy',
    'colour': 'colour-science',
}

missing_packages = [
    pip_name
    for import_name, pip_name in required_packages.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_packages:
    print('Installing missing package(s):', ', '.join(missing_packages))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('All required packages are already installed.')

# Imports required by all subsequent cells.
import re
import numpy as np
import colour
from colour.adaptation import chromatic_adaptation_VonKries

np.set_printoptions(precision=8, suppress=True)
print('Python version:', sys.version.split()[0])
print('NumPy version:', np.__version__)
print('colour-science version:', colour.__version__)

Installing missing package(s): colour-science
Python version: 3.12.13
NumPy version: 2.0.2
colour-science version: 0.4.7


In [2]:
#@title 2. Define input and colourimetric settings
# Enter any chromatic Munsell notation accepted by colour-science.
MUNSELL_INPUT = '10B 9/2'  #@param {type:'string'}

# Use a consistent observer for CIE white points and Lab calculations.
OBSERVER = 'CIE 1931 2 Degree Standard Observer'

# Bradford is a transform in the Von Kries chromatic-adaptation family.
CAT_TRANSFORM = 'Bradford'

# Munsell renotation xyY values are defined under Illuminant C.
# The requested Lab output is defined relative to Illuminant D65.
xy_C = colour.CCS_ILLUMINANTS[OBSERVER]['C']
xy_D65 = colour.CCS_ILLUMINANTS[OBSERVER]['D65']

# CAT white points use XYZ coordinates with Y normalised to 1.
XYZ_w_C = colour.xy_to_XYZ(xy_C)
XYZ_w_D65 = colour.xy_to_XYZ(xy_D65)

MUNSELL_INPUT = re.sub(r'\s+', ' ', MUNSELL_INPUT.strip().upper())

print('Munsell input:', MUNSELL_INPUT)
print('Illuminant C xy:', xy_C)
print('Illuminant D65 xy:', xy_D65)
print('Chromatic adaptation: Von Kries with', CAT_TRANSFORM, 'transform')

Munsell input: 10B 9/2
Illuminant C xy: [ 0.31006  0.31616]
Illuminant D65 xy: [ 0.3127  0.329 ]
Chromatic adaptation: Von Kries with Bradford transform


In [3]:
#@title 3. Define Munsell hue-boundary normalisation
# The standard Munsell hue sectors are arranged cyclically in this order.
HUE_SECTORS = ['R', 'YR', 'Y', 'GY', 'G', 'BG', 'B', 'PB', 'P', 'RP']

def format_munsell_number(number):
    """Format an integer-like Munsell hue component without a trailing decimal zero."""
    number = float(number)
    if np.isclose(number, round(number)):
        return str(int(round(number)))
    return f'{number:.2f}'.rstrip('0').rstrip('.')

def normalise_zero_hue_to_previous_sector(munsell_notation, tolerance=1e-8):
    """
    Change 0.0<sector> V/C to 10<preceding sector> V/C.

    Examples:
    0.0PB 9/2 becomes 10B 9/2.
    0R 7/6 becomes 10RP 7/6.

    Value and chroma are retained exactly as returned by the inverse conversion.
    Nonzero hue numbers are retained, apart from harmless display formatting.
    """
    text = str(munsell_notation).strip().upper()
    match = re.fullmatch(
        r'([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*([A-Z]+)\s+([^\s/]+)\s*/\s*([^\s]+)',
        text,
    )
    if match is None:
        raise ValueError(f'Cannot parse Munsell notation: {munsell_notation!r}')

    hue_number_text, sector, value_text, chroma_text = match.groups()
    hue_number = float(hue_number_text)

    if np.isclose(hue_number, 0.0, atol=tolerance) and sector in HUE_SECTORS:
        prior_sector = HUE_SECTORS[(HUE_SECTORS.index(sector) - 1) % len(HUE_SECTORS)]
        return f'10{prior_sector} {value_text}/{chroma_text}'

    return f'{format_munsell_number(hue_number)}{sector} {value_text}/{chroma_text}'

for example in ['0.0PB 9/2', '0B 5/4', '0R 7/6', '2.5PB 9/2']:
    print(f'{example} -> {normalise_zero_hue_to_previous_sector(example)}')

0.0PB 9/2 -> 10B 9/2
0B 5/4 -> 10BG 5/4
0R 7/6 -> 10RP 7/6
2.5PB 9/2 -> 2.5PB 9/2


In [4]:
#@title 4. Convert Munsell under C to CIE Lab under D65
# Step 1: Munsell notation to standard Munsell renotation xyY under Illuminant C.
xyY_C = colour.munsell_colour_to_xyY(MUNSELL_INPUT)

# Step 2: xyY(C) to XYZ(C).
XYZ_C = colour.xyY_to_XYZ(xyY_C)

# Step 3: Adapt C-referenced XYZ to D65-referenced XYZ.
XYZ_D65 = chromatic_adaptation_VonKries(
    XYZ_C,
    XYZ_w_C,
    XYZ_w_D65,
    transform=CAT_TRANSFORM,
)

# Step 4: Calculate CIE Lab relative to D65.
Lab_D65 = colour.XYZ_to_Lab(XYZ_D65, illuminant=xy_D65)

print('MUNSELL C TO LAB D65')
print('Munsell input under C:', MUNSELL_INPUT)
print('xyY under C:', xyY_C)
print('XYZ under C:', XYZ_C)
print('XYZ adapted to D65:', XYZ_D65)
print('CIE Lab under D65:', Lab_D65)

MUNSELL C TO LAB D65
Munsell input under C: 10B 9/2
xyY under C: [ 0.2949      0.3076      0.76695586]
XYZ under C: [ 0.73529026  0.76695586  0.99110844]
XYZ adapted to D65: [ 0.7112753   0.76691891  0.91281028]
CIE Lab under D65: [ 90.17961294  -3.72363963  -5.50196155]


In [5]:
#@title 5. Convert CIE Lab under D65 back to Munsell under C
# Lab_D65 becomes the new input to the inverse workflow.
# To use an independently measured Lab(D65) triplet, replace the next line.
# Example: Lab_D65_input = np.array([90.0, -10.0, -15.0])
Lab_D65_input = np.asarray(Lab_D65, dtype=float)

# Step 5: Lab(D65) to XYZ(D65).
XYZ_D65_from_Lab = colour.Lab_to_XYZ(Lab_D65_input, illuminant=xy_D65)

# Step 6: Adapt D65-referenced XYZ back to C-referenced XYZ.
# This ensures that the inverse Munsell calculation receives xyY under C.
XYZ_C_restored = chromatic_adaptation_VonKries(
    XYZ_D65_from_Lab,
    XYZ_w_D65,
    XYZ_w_C,
    transform=CAT_TRANSFORM,
)

# Step 7: XYZ(C) to xyY(C), then xyY(C) to interpolated Munsell notation.
xyY_C_restored = colour.XYZ_to_xyY(XYZ_C_restored)
MUNSELL_RESTORED_RAW = colour.xyY_to_munsell_colour(xyY_C_restored)

# Apply the requested 0.0-sector boundary convention.
MUNSELL_RESTORED = normalise_zero_hue_to_previous_sector(MUNSELL_RESTORED_RAW)

print('LAB D65 TO MUNSELL C')
print('Lab(D65) input:', Lab_D65_input)
print('XYZ reconstructed under D65:', XYZ_D65_from_Lab)
print('XYZ restored under C:', XYZ_C_restored)
print('xyY restored under C:', xyY_C_restored)
print('Raw restored Munsell(C):', MUNSELL_RESTORED_RAW)
print('Boundary-normalised Munsell(C):', MUNSELL_RESTORED)

LAB D65 TO MUNSELL C
Lab(D65) input: [ 90.17961294  -3.72363963  -5.50196155]
XYZ reconstructed under D65: [ 0.7112753   0.76691891  0.91281028]
XYZ restored under C: [ 0.73529026  0.76695586  0.99110844]
xyY restored under C: [ 0.2949      0.3076      0.76695586]
Raw restored Munsell(C): 0.0PB 9.0/2.0
Boundary-normalised Munsell(C): 10B 9.0/2.0


In [6]:
#@title 6. Calculate numerical round-trip diagnostics
# Reconstruct Lab(D65) from XYZ(D65) for a numerical consistency check.
Lab_D65_reconstructed = colour.XYZ_to_Lab(
    XYZ_D65_from_Lab,
    illuminant=xy_D65,
)

# Use the public and current colour-science Delta E interface.
delta_E00 = float(
    colour.delta_E(
        Lab_D65_input,
        Lab_D65_reconstructed,
        method='CIE 2000',
    )
)

print('ROUND-TRIP DIAGNOSTICS')
print('Delta XYZ under C:', XYZ_C_restored - XYZ_C)
print('Delta xyY under C:', xyY_C_restored - xyY_C)
print('Maximum absolute Delta XYZ under C:', np.max(np.abs(XYZ_C_restored - XYZ_C)))
print('Delta E00 between Lab input and reconstructed Lab:', delta_E00)
print('Final restored Munsell under C:', MUNSELL_RESTORED)

ROUND-TRIP DIAGNOSTICS
Delta XYZ under C: [ 0. -0.  0.]
Delta xyY under C: [ 0. -0. -0.]
Maximum absolute Delta XYZ under C: 3.33066907388e-16
Delta E00 between Lab input and reconstructed Lab: 0.0
Final restored Munsell under C: 10B 9.0/2.0


## Notes

The first code cell imports all libraries needed later in the notebook. It checks import availability before invoking pip, so no installation is attempted when NumPy and colour-science are already available.

The hue-boundary correction is cyclic: 0PB becomes 10B, 0B becomes 10BG, and 0R becomes 10RP. It only changes a hue that is numerically zero within the selected tolerance. Value and chroma are preserved.

Inverse Munsell conversion is interpolative. Therefore, a result can differ slightly from the original Munsell notation even if the XYZ and Lab numerical round trip is effectively exact.